## <span style="color: #ff7402">__Atelier Préparation Données Images__</span>

Contexte 
Une entreprise souhaite développer un système d’intelligence artificielle capable de reconnaître 
automatiquement le type de déchet présent sur une photographie afin d'améliorer le tri des 
déchets. 

Le modèle devra classer chaque image dans l'une des catégories suivantes : 

* cardboard : cartons ondulés, cartons plats, … 
* plastic : bouteilles, emballages plastiques... 
* paper : feuilles,  journaux... 
* glass : bouteilles et objets en verre... 
* metal : canettes, boîtes métalliques... 
* trash : emballages bonbons, tasses jetables, ... 

Le problème est que les images collectées proviennent de plusieurs sources. Elles ne sont donc pas 
homogènes : dimensions différentes ; formats différents ; images RGB et grayscale ; certaines images 
sont trop petites ; certaines images sont corrompues ; quelques images sont vides ; images 
dupliquées ; quelques images placées dans le mauvais dossier ; classes déséquilibrées.

L'objectif de l'atelier est donc de construire un jeu de données images propre et homogène, prêt à 
être utilisé pour entraîner un modèle de Machine Learning ou de Deep Learning. 

### <span style="color: #ffe602">__Partie 1 – Exploration du dataset__</span>

Un programme Python capable de récupérer, pour chaque image, son nom, sa classe, 
son format, son mode, sa largeur, sa hauteur, l’écart-type de ses pixels, son nombre de canaux et sa 
taille.

Construction d'un script simple pour auditer le dossier data/raw/ et extraire les caractéristiques techniques de chaque image avec la bibliothèque standard de gestion de fichiers en Python (os ou pathlib) et la bibliothèque de traitement d'images incontournable : Pillow (PIL).

In [1]:
import os
import pandas as pd
from PIL import Image

# 1. Définition du chemin vers les données brutes
CHEMIN_RAW = "../data/raw"

# 2. Liste pour stocker les informations de chaque image
liste_informations = []

# 3. Parcours des dossiers de chaque classe dans data/raw
for nom_classe in os.listdir(CHEMIN_RAW):
    chemin_classe = os.path.join(CHEMIN_RAW, nom_classe)
    
    # On s'assure qu'on parcourt bien un dossier (ex: cardboard, glass...)
    if os.path.isdir(chemin_classe):
        
        # Parcours de chaque fichier dans le dossier de la classe
        for nom_fichier in os.listdir(chemin_classe):
            chemin_image = os.path.join(chemin_classe, nom_fichier)
            
            # Pour éviter de traiter des fichiers système cachés (ex: .DS_Store)
            if nom_fichier.startswith('.'):
                continue
                
            # Initialisation d'un dictionnaire avec les infos de base
            info_image = {
                "nom": nom_fichier,
                "classe": nom_classe,
                "format": None,
                "mode": None,
                "largeur": None,
                "hauteur": None,
                "ecart_type": None,
                "canaux": None,
                "taille_octets": os.path.getsize(chemin_image), # Taille du fichier sur le disque
                "statut": "Valide"
            }
            
            try:
                # Tentative d'ouverture de l'image avec Pillow
                with Image.open(chemin_image) as img:
                    # On force le chargement pour vérifier si le fichier est réellement corrompu
                    img.verify() 
                    
                # On réouvre l'image pour lire ses propriétés (verify() ferme le fichier)
                with Image.open(chemin_image) as img:
                    info_image["format"] = img.format  # Ex: JPEG, PNG
                    info_image["mode"] = img.mode      # Ex: RGB, L (Grayscale)
                    info_image["largeur"] = img.size[0]
                    info_image["hauteur"] = img.size[1]
                    
                    # Détermination du nombre de canaux selon le mode
                    # RGB = 3 canaux, RGBA = 4 canaux, L (noir et blanc) = 1 canal
                    info_image["canaux"] = len(img.getbands())
                    
                    # Pour l'écart-type, on convertit temporairement en niveaux de gris 
                    # afin d'avoir une seule valeur globale simple pour cette étape
                    statistiques = img.convert("L").getextrema() 
                    # Note : getbands/histogram peut donner l'écart-type, mais pour faire simple
                    # et efficace sans numpy à ce stade, on peut aussi charger les données en niveaux de gris
                    import math
                    stat_visuelles = Image.Image.getdata(img.convert("L"))
                    # Version simplifiée pour débutant sans charger de grosse matrice :
                    # On utilise l'astuce de stocker temporairement la variance ou une valeur par défaut
                    # Mais pour être précis et performant, on peut importer brièvement numpy ici :
                    import numpy as np
                    matrice_pixels = np.array(img)
                    info_image["ecart_type"] = round(float(np.std(matrice_pixels)), 2)

            except Exception as e:
                # Si une erreur survient, le fichier est marqué comme corrompu
                info_image["statut"] = "Corrompu"
            
            # Ajout des données de l'image actuelle à notre liste
            liste_informations.append(info_image)

# 4. Conversion de la liste en DataFrame Pandas pour visualiser le résultat
df_exploration = pd.DataFrame(liste_informations)

# Affichage des 10 premières lignes du résultat
print("Aperçu des données explorées :")
print(df_exploration.head(10))


C:\Users\cissc\AppData\Local\Temp\ipykernel_19408\2156876818.py:63: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  stat_visuelles = Image.Image.getdata(img.convert("L"))


Aperçu des données explorées :
                nom     classe format mode  largeur  hauteur  ecart_type  \
0    cardboard1.jpg  cardboard   JPEG  RGB    512.0    384.0       40.59   
1   cardboard10.jpg  cardboard   JPEG  RGB    512.0    384.0       42.57   
2  cardboard100.jpg  cardboard   JPEG  RGB    512.0    384.0       46.11   
3  cardboard101.jpg  cardboard   JPEG  RGB    512.0    384.0       72.26   
4  cardboard102.jpg  cardboard   JPEG  RGB    512.0    384.0       48.39   
5  cardboard103.jpg  cardboard   JPEG  RGB    512.0    384.0       40.74   
6  cardboard104.jpg  cardboard   JPEG  RGB    512.0    384.0       38.82   
7  cardboard105.jpg  cardboard   JPEG  RGB    512.0    384.0       49.68   
8  cardboard106.jpg  cardboard   JPEG  RGB    512.0    384.0       57.07   
9  cardboard107.jpg  cardboard   JPEG  RGB    512.0    384.0       41.68   

   canaux  taille_octets  statut  
0     3.0          17333  Valide  
1     3.0          21683  Valide  
2     3.0          14884  V

### <span style="color: #ffde07">__Partie 2 – Détection des images corrompues__</span> 

Une fonction qui détecte une image corrompue. 

Une image est considérée comme corrompue lorsque le fichier est endommagé (téléchargement incomplet, bug d'écriture sur le disque, format non reconnu).

In [2]:
# Fonction de detection d'image corrompue

from PIL import Image

def est_image_corrompue(chemin_fichier):
    """
    Vérifie si une image est corrompue ou illisible.
    Renvoie True si l'image est corrompue, False sinon.
    """
    try:
        with Image.open(chemin_fichier) as img:
            # La méthode verify() vérifie la structure du fichier sans décoder les pixels
            img.verify()
        return False  # Si aucune erreur n'est levée, l'image n'est pas corrompue
    except Exception:
        return True   # Si une erreur survient, l'image est corrompue


In [3]:
# Utilisation de la fonction de detection pour vérifier les images dans notre dossier raw

# Chemin vers nos images brutes
CHEMIN_RAW = "../data/raw"

# Liste pour stocker les chemins des images corrompues trouvées
images_corrompues = []

# Parcours des sous-dossiers de classes
for nom_classe in os.listdir(CHEMIN_RAW):
    chemin_classe = os.path.join(CHEMIN_RAW, nom_classe)
    
    if os.path.isdir(chemin_classe):
        for nom_fichier in os.listdir(chemin_classe):
            # Ignorer les fichiers système cachés
            if nom_fichier.startswith('.'):
                continue
                
            chemin_complet = os.path.join(chemin_classe, nom_fichier)
            
            # Appel de notre fonction
            if est_image_corrompue(chemin_complet):
                images_corrompues.append({
                    "Nom": nom_fichier,
                    "Classe": nom_classe,
                    "Chemin": chemin_complet
                })

# Affichage du bilan
print(f"Nombre total d'images corrompues détectées : {len(images_corrompues)}")
if len(images_corrompues) > 0:
    print("\nListe des images corrompues :")
    for img in images_corrompues:
        print(f"- Classe [{img['Classe']}] : Fichier {img['Nom']}")


Nombre total d'images corrompues détectées : 6

Liste des images corrompues :
- Classe [cardboard] : Fichier cardboard83.jpg
- Classe [glass] : Fichier glass74.jpg
- Classe [metal] : Fichier metal48.jpg
- Classe [paper] : Fichier paper213.jpg
- Classe [plastic] : Fichier plastic13.jpg
- Classe [trash] : Fichier trash3.jpg


### <span style="color: #e2c717">__Partie 3 – Détection des images vides__</span> 

Écrire et se servir d’une fonction qui détecte les images vides : image entièrement noire, image 
entièrement blanche ou image dont les pixels présentent très peu de variation. 

Une image est considérée comme vide ou quasi vide si tous ses pixels ont la même couleur ou si leurs nuances sont extrêmement proches.

Pour mesurer cela de manière scientifique et simple, on utilise l'écart-type (standard deviation ou std en anglais) des pixels :

* Si l'image est entièrement noire ou entièrement blanche, l'écart-type est strictement égal à 0 (aucune variation).

* Si l'image contient un léger bruit de fond ou une couleur unie imparfaite (un fond gris ou blanc continu), l'écart-type sera très proche de 0 (très peu de variation).

In [4]:
# Fonction de detection des images vides

def est_image_vide(chemin_fichier, seuil_variation=2.0):
    """
    Détecte si une image est entièrement noire, blanche ou presque uniforme.
    Se base sur l'écart-type des pixels en niveaux de gris.
    
    seuil_variation : Écart-type en dessous duquel l'image est considérée comme vide.
    """
    try:
        with Image.open(chemin_fichier) as img:
            # 1. On convertit l'image en niveaux de gris ('L') pour avoir une seule matrice d'analyse
            img_gris = img.convert('L')
            
            # 2. On transforme l'image en tableau de nombres (Numpy array)
            tableau_pixels = np.array(img_gris)
            
            # 3. On calcule l'écart-type de toutes les valeurs du tableau
            ecart_type = np.std(tableau_pixels)
            
            # 4. Si la variation est inférieure au seuil choisi, l'image est jugée vide
            if ecart_type < seuil_variation:
                return True
            else:
                return False
                
    except Exception:
        # Si l'image est corrompue et impossible à ouvrir, on ne la traite pas ici 
        # (elle a déjà été gérée dans la Partie 2)
        return False


In [5]:
# Utilisation de la fonction de detection pour vérifier les images dans notre dossier raw

CHEMIN_RAW = "../data/raw"
images_vides = []

# Parcours des sous-dossiers
for nom_classe in os.listdir(CHEMIN_RAW):
    chemin_classe = os.path.join(CHEMIN_RAW, nom_classe)
    
    if os.path.isdir(chemin_classe):
        for nom_fichier in os.listdir(chemin_classe):
            if nom_fichier.startswith('.'):
                continue
                
            chemin_complet = os.path.join(chemin_classe, nom_fichier)
            
            # On applique notre fonction de détection
            if est_image_vide(chemin_complet, seuil_variation=2.0):
                images_vides.append({
                    "Nom": nom_fichier,
                    "Classe": nom_classe,
                    "Chemin": chemin_complet
                })

# Bilan de la détection
print(f"Nombre total d'images vides ou uniformes détectées : {len(images_vides)}")
if len(images_vides) > 0:
    print("\nListe des images problématiques trouvées :")
    for img in images_vides:
        print(f"- Classe [{img['Classe']}] : Fichier {img['Nom']}")


Nombre total d'images vides ou uniformes détectées : 4

Liste des images problématiques trouvées :
- Classe [cardboard] : Fichier image-blanche-512x384.jpg
- Classe [glass] : Fichier image-noire-512x384.png
- Classe [metal] : Fichier image-blanche-512x384.jpg
- Classe [metal] : Fichier image-noire-512x384.png


### <span style="color: #dcc00d">__Partie 4 – Détecter les différences de résolution__</span> 

1) Déterminer la résolution minimale, la résolution maximale, les résolutions les plus 
fréquentes et le nombre d'images par résolution.

Puisqu'on a déjà extrait les informations de base lors de la Partie 1 dans un tableau Pandas (df_exploration), on va réutiliser ce tableau pour faire nos calculs de manière très simple et rapide.

Note : On prend soin de filtrer notre tableau pour ne travailler que sur les images "Valides" (en excluant les images corrompues qui n'ont pas de dimensions).

In [ ]:
# 1. On filtre le DataFrame de la Partie 1 pour ne garder que les images valides
df_valides = df_exploration[df_exploration["statut"] == "Valide"].copy()

# 2. Création d'une colonne "resolution" sous forme de texte (ex: "800x600") pour faciliter les calculs
df_valides["resolution"] = df_valides.apply(lambda row: f"{row['largeur']}x{row['hauteur']}", axis=1)

# 3. Calcul des résolutions minimales et maximales
# On cherche l'image avec la plus petite et la plus grande surface (largeur x hauteur)
df_valides["surface"] = df_valides["largeur"] * df_valides["hauteur"]

id_min = df_valides["surface"].idxmin()
id_max = df_valides["surface"].idxmax()

res_minimale = df_valides.loc[id_min, "resolution"]
res_maximale = df_valides.loc[id_max, "resolution"]

# 4. Compter le nombre d'images par résolution et identifier les plus fréquentes
distribution_res = df_valides["resolution"].value_counts()

# AFFICHAGE DES RÉSULTATS 
print("BILAN DE LA RÉSOLUTION DES IMAGES")
print(f"Résolution minimale trouvée : {res_minimale} (Surface : {df_valides.loc[id_min, 'surface']} px)")
print(f"Résolution maximale trouvée : {res_maximale} (Surface : {df_valides.loc[id_max, 'surface']} px)")
print("\nDISTRIBUTION DES RÉSULTATIONS (Nombre d'images par résolution)")
print(distribution_res)

print("\nLES RÉSUTIONS LES PLUS FRÉQUENTES (Top 3)")
print(distribution_res.head(3))


BILAN DE LA RÉSOLUTION DES IMAGES
Résolution minimale trouvée : 32.0x32.0 (Surface : 1024.0 px)
Résolution maximale trouvée : 512.0x384.0 (Surface : 196608.0 px)

DISTRIBUTION DES RÉSULTATIONS (Nombre d'images par résolution)
resolution
512.0x384.0    1013
32.0x32.0         5
48.0x32.0         4
40.0x40.0         4
Name: count, dtype: int64

LES RÉSUTIONS LES PLUS FRÉQUENTES (Top 3)
resolution
512.0x384.0    1013
32.0x32.0         5
48.0x32.0         4
Name: count, dtype: int64


2) On décide qu'une image doit avoir au minimum 64 × 64 pixels. 

On définit donc un seuil strict : si la largeur est inférieure à 64 OU si la hauteur est inférieure à 64, l'image est rejetée.

Identifions toutes les images ne respectant pas cette contrainte.

In [8]:
# Identification des images trop petites

# 1. On filtre pour n'analyser que les images valides
df_valides = df_exploration[df_exploration["statut"] == "Valide"].copy()

# 2. Application du filtre : largeur < 64 OU hauteur < 64
df_trop_petites = df_valides[(df_valides["largeur"] < 64) | (df_valides["hauteur"] < 64)]

# 3. Affichage du bilan
print("DÉTECTION DES IMAGES TROP PETITES (< 64x64 px) ")
print(f"Nombre d'images ne respectant pas la contrainte : {len(df_trop_petites)}")

if len(df_trop_petites) > 0:
    print("\nListe des images trop petites à exclure :")
    # On affiche le nom, la classe et les dimensions réelles pour vérification
    print(df_trop_petites[["nom", "classe", "largeur", "hauteur"]].to_string(index=False))
else:
    print("\nBonne nouvelle ! Toutes les images valides font au moins 64x64 pixels.")


DÉTECTION DES IMAGES TROP PETITES (< 64x64 px) 
Nombre d'images ne respectant pas la contrainte : 13

Liste des images trop petites à exclure :
             nom    classe  largeur  hauteur
cardboard117.jpg cardboard     48.0     32.0
 cardboard22.jpg cardboard     32.0     32.0
 cardboard70.jpg cardboard     40.0     40.0
    glass100.jpg     glass     40.0     40.0
     glass15.jpg     glass     48.0     32.0
     glass21.jpg     glass     32.0     32.0
     glass23.jpg     glass     32.0     32.0
    metal121.jpg     metal     48.0     32.0
      metal2.jpg     metal     32.0     32.0
     metal26.jpg     metal     40.0     40.0
     paper10.jpg     paper     32.0     32.0
     paper54.jpg     paper     40.0     40.0
     paper64.jpg     paper     48.0     32.0


### <span style="color: #fe0d7a">__Partie 5 – Détection des différents canaux__</span> 

#### <span style="color: #f04793">Détermination du nombre d'images selon leur nombre de canaux.</span> 

On va à nouveau exploiter notre tableau Pandas df_exploration de la Partie 1. On va filtrer sur les images valides et utiliser la fonction .value_counts() sur la colonne canaux.

In [11]:
# 1. On ne garde que les images valides
df_valides = df_exploration[df_exploration["statut"] == "Valide"].copy()

# 2. On compte le nombre d'images pour chaque quantité de canaux
distribution_canaux = df_valides["canaux"].value_counts()

# 3. On extrait aussi la distribution selon le mode de l'image (RGB, L, RGBA...) pour plus de précision
distribution_modes = df_valides["mode"].value_counts()

# AFFICHAGE DES RÉSULTATS 
print("ANALYSE DES CANAUX DE COULEUR ")
print("Nombre d'images selon leur nombre de canaux :")
for canaux, nb_images in distribution_canaux.items():
    print(f"- {int(canaux)} canal/canaux : {nb_images} image(s)")

print("\nDÉTAIL PAR MODE PIL")
for mode, nb_images in distribution_modes.items():
    print(f"- Mode '{mode}' : {nb_images} image(s)")


ANALYSE DES CANAUX DE COULEUR 
Nombre d'images selon leur nombre de canaux :
- 3 canal/canaux : 1006 image(s)
- 4 canal/canaux : 18 image(s)
- 1 canal/canaux : 2 image(s)

DÉTAIL PAR MODE PIL
- Mode 'RGB' : 1006 image(s)
- Mode 'RGBA' : 18 image(s)
- Mode 'P' : 2 image(s)
